# Statistical Inference & Confidence Intervals

Companion notebook for the [Statistical Inference lesson](https://ml-viz-ruby.vercel.app/courses/probability-statistics/06-statistical-inference).

We make the abstractions concrete by **simulation**: watch the Central Limit Theorem turn a skewed
population into normal sample means, confirm the standard error shrinks like 1/√n, and check that a
95% confidence interval really does cover the truth ~95% of the time. Pure NumPy + Matplotlib.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(0)

## Intuition — what a sample can tell you about the world

You never see the whole population — only a sample — yet you need to state something about
the population *and* how much to trust it. Statistical inference is that bridge. Three facts
do the heavy lifting, and all three are visible by simulation: the **Central Limit Theorem**
(sample means go bell-shaped whatever the population looks like), the **standard error**
`σ/√n` (how much a sample mean wobbles, shrinking only as `√n`), and the **confidence
interval** (a range that traps the true value a known fraction of the time). We build each by
brute-force simulation, then reach for `scipy` for the exact intervals.

## 1 — The Central Limit Theorem in action

We draw from a strongly skewed (exponential) population. For each sample size n we take many samples,
record each sample's mean, and histogram them. The skew washes out and the means become bell-shaped
as n grows — regardless of the population's shape.

In [ ]:
pop_mean = 1.0          # exponential(scale=1) has mean 1 and SD 1
pop_sd = 1.0
fig, axes = plt.subplots(1, 4, figsize=(12, 3), sharey=True)
for ax, n in zip(axes, [1, 2, 10, 50]):
    means = rng.exponential(scale=1.0, size=(5000, n)).mean(axis=1)
    ax.hist(means, bins=40, color='#6366f1', density=True)
    ax.axvline(pop_mean, color='#fb7185', ls='--')
    ax.set_title(f'n = {n}'); ax.set_xlim(0, 3)
axes[0].set_ylabel('density')
fig.suptitle('Sampling distribution of the mean approaches normal as n grows')
plt.tight_layout(); plt.show()

**What to notice:** the population is a hard-skewed exponential, yet by `n = 50` the
distribution of the *sample mean* is a clean bell centered on the true mean. That's the CLT:
averaging normalizes, which is why so much of statistics assumes normal sampling distributions
even for non-normal data.

## 2 — Standard error shrinks like 1/√n

The theoretical standard error is SE = σ/√n. We compare it to the *measured* spread of sample means
across many simulated samples — they should match closely.

In [ ]:
ns = np.array([1, 2, 5, 10, 25, 50, 100, 200])
theoretical = pop_sd / np.sqrt(ns)
measured = [rng.exponential(1.0, size=(4000, n)).mean(axis=1).std() for n in ns]

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(ns, theoretical, 'o-', color='#2dd4bf', label='theoretical  σ/√n')
ax.plot(ns, measured, 's--', color='#fb7185', label='measured spread of means')
ax.set_xlabel('sample size n'); ax.set_ylabel('standard error')
ax.set_title('Standard error follows σ/√n: 4x the data halves the error')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"n=25  -> SE = {pop_sd/np.sqrt(25):.3f}")
print(f"n=100 -> SE = {pop_sd/np.sqrt(100):.3f}  (4x the data -> half the SE)")

**What to notice:** the measured spread of sample means tracks the theoretical `σ/√n`
almost exactly, and the `√n` is the punchline — going from `n=25` to `n=100` (4× the data)
only **halves** the error. Precision is expensive: each extra digit of accuracy costs ~100×
more samples.

## 3 — Do 95% confidence intervals really cover 95%?

We build a 95% CI (`x̄ ± 1.96·SE`) from many independent samples of a normal population and count how
often the interval contains the *true* mean. The coverage should land near 95% — that long-run
coverage is exactly what '95% confidence' means.

In [ ]:
true_mu, true_sigma, n, trials = 5.0, 2.0, 40, 10000
z_star = 1.96
covered = 0
for _ in range(trials):
    sample = rng.normal(true_mu, true_sigma, size=n)
    xbar = sample.mean()
    se = sample.std(ddof=1) / np.sqrt(n)
    lo, hi = xbar - z_star * se, xbar + z_star * se
    if lo <= true_mu <= hi:
        covered += 1
print(f"empirical coverage of nominal 95% CI: {100*covered/trials:.1f}%  (target 95%)")

**What to notice:** across 10,000 simulated samples the 95% interval contains the true mean
~95% of the time. That's the *actual* meaning of "95% confidence" — a property of the
**procedure** over many repeats, not a probability about any single interval (a gotcha
below).

## 4. The library way — exact intervals with `scipy.stats`

`scipy.stats.norm.interval` builds the interval directly from a mean and standard error, and
`scipy.stats.t.interval` gives the proper **t-interval** for when σ is estimated from the data
(the honest choice at small `n`). The cell checks the normal interval matches `x̄ ± 1.96·SE`.

In [ ]:
from scipy import stats

sample = rng.normal(5.0, 2.0, size=40)
xbar = sample.mean()
se = sample.std(ddof=1) / np.sqrt(40)

lo_z, hi_z = stats.norm.interval(0.95, loc=xbar, scale=se)   # z-interval (sigma known)
lo_t, hi_t = stats.t.interval(0.95, df=39, loc=xbar, scale=se)  # t-interval (sigma estimated)

print(f'z-interval: ({lo_z:.3f}, {hi_z:.3f})')
print(f't-interval: ({lo_t:.3f}, {hi_t:.3f})   (slightly wider — honest about estimating sigma)')
assert np.allclose([lo_z, hi_z], [xbar - 1.96*se, xbar + 1.96*se], atol=0.01), "norm.interval == xbar +/- 1.96 SE"
print('\nscipy.stats.norm.interval matches xbar +/- 1.96*SE ✓')

**What to notice:** `scipy.stats.norm.interval` reproduces the hand `x̄ ± 1.96·SE`, and the
**t-interval is a touch wider** — the correct penalty for having estimated σ from the sample.
At small `n` you should use `t`; by `n≈100` the two nearly coincide.

## 5. Gotchas & limitations

- **A CI is about the procedure, not the parameter.** "95% confidence" means 95% of such
  intervals cover the truth — *not* "95% probability the mean is in this one." The true mean is
  fixed; the interval is random.
- **z vs t.** Use the normal `z` only when σ is known (rare); otherwise use the **t-distribution**
  (heavier tails) — especially at small `n`.
- **`√n` is slow.** Halving error needs 4× the data; this caps how precise any study can
  affordably be.
- **The CLT needs finite variance and enough `n`.** Heavy tails or tiny samples break the
  normal approximation.

In [ ]:
# z vs t critical values: t is wider at small n, converges to z as n grows
from scipy import stats
for n in [5, 20, 100, 1000]:
    t_star = stats.t.ppf(0.975, df=n-1)
    print(f'n={n:>4}: t* = {t_star:.3f}  vs  z* = 1.960   (gap {t_star-1.96:+.3f})')

**What to notice:** the t critical value is `2.78` at `n=5` but essentially `1.96` by
`n=1000` — the small-sample penalty for not knowing σ fades as data accumulates. Using `z`
at `n=5` would give an interval that's too *narrow*, overstating your certainty.

## Key takeaways

- **CLT:** sample means become normal regardless of the population — the basis for most
  inference.
- **Standard error `σ/√n`** shrinks only as `√n`: 4× data → half the error.
- **Confidence intervals** cover the truth at their nominal rate over many repeats (a
  procedure property); use the **t-interval** when σ is estimated.
- In practice: `scipy.stats.norm.interval` / `t.interval`, and mind the CI interpretation,
  z-vs-t, and the `√n` cost.

**Next:** [Hypothesis Testing](https://ml-viz-ruby.vercel.app/courses/probability-statistics/07-hypothesis-testing).

## ✏️ Your turn

**Exercise.** Implement `standard_error(sigma, n)` and `confidence_interval(xbar, sigma, n, z_star)`
returning the `(low, high)` tuple for a CI of the mean. From the lesson: `SE = σ/√n` and the interval
is `x̄ ± z*·SE`.

In [ ]:
def standard_error(sigma, n):
    # TODO(you): return the standard error of the mean
    return ...

def confidence_interval(xbar, sigma, n, z_star=1.96):
    # TODO(you): return (low, high) for x̄ ± z*·SE
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert standard_error(20, 100) == 2.0
assert abs(standard_error(20, 64) - 2.5) < 1e-9
lo, hi = confidence_interval(50, 20, 100)        # SE = 2, margin = 3.92
assert abs(lo - 46.08) < 1e-6 and abs(hi - 53.92) < 1e-6
print("\u2713 SE and confidence interval are correct")

<details>
<summary>Solution</summary>

```python
def standard_error(sigma, n):
    return sigma / np.sqrt(n)

def confidence_interval(xbar, sigma, n, z_star=1.96):
    se = standard_error(sigma, n)
    return (xbar - z_star * se, xbar + z_star * se)
```

The √n in the standard error is the central fact of inference: precision improves only as the square
root of the data, so each extra digit of accuracy costs disproportionately more samples.

</details>